# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [1]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-02','lines':[{'item':'Hot Dog','qty':3}]},
]
#changed vendor_id to V-02 for the hot dog so that one of the vendors in the orders is not in the vendor list
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

**One row of my result is one:** _..._

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [2]:
import pandas as pd
#flattening the orders_json
lines = pd.json_normalize(orders_json, record_path='lines', meta=['order','vendor_id'])
assert len(lines) == 4
assert lines['qty'].sum() == 7

print (lines)
# TODO: pd.json_normalize(orders_json, record_path='lines', meta=[...])
# TODO: assert the row count and the quantity total

           item  qty order vendor_id
0  Cheeseburger    2     1      V-01
1          Soda    1     1      V-01
2   Foam Finger    1     2      V-10
3       Hot Dog    3     3      V-02


### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [3]:
#converting vendors_json into a DataFrame
vendors_df = pd.DataFrame(vendors_json)
#left join orders_df with vendors_df w/ indicator = True
joined = pd.merge(lines, vendors_df, on='vendor_id', how='left', indicator=True)
#print
print(joined)
#identifying unmatches vendors + attached units
unmatched = joined[joined['_merge'] == 'left_only']
unmatched_vendors = unmatched['vendor_id'].unique()
unmatched_units = unmatched['qty'].sum()
print(f"Unmatched vendors: {unmatched_vendors}")
print(f"Unmatched units: {unmatched_units}")


           item  qty order vendor_id          name zone     _merge
0  Cheeseburger    2     1      V-01  Hoos Burgers    A       both
1          Soda    1     1      V-01  Hoos Burgers    A       both
2   Foam Finger    1     2      V-10     Cav Merch    A       both
3       Hot Dog    3     3      V-02           NaN  NaN  left_only
Unmatched vendors: ['V-02']
Unmatched units: 3


### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [4]:
joined['name'] = joined['name'].fillna('Unknown Vendor')
total_quantity = (joined.groupby(['vendor_id', 'name'])['qty']
    .sum()
    .reset_index())
print(total_quantity)

#I edited the original JSON object with the vendor IDs so that the vendor selling the hot
#dogs would be a separate vendor and thus not defined in vendors_json. This allowed for
#unmatched vendors and units to yield a result in Q2. Then this unmatched vendor appears
#as its own row within Q3's table labeled "Unknown Vendor." I'm not totally sure if this
#was the right way to do it, but I tried to create an unmatched vendor that was reported
#in this table


  vendor_id            name  qty
0      V-01    Hoos Burgers    3
1      V-02  Unknown Vendor    3
2      V-10       Cav Merch    1


### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [5]:
#grouping by zone and calculating quantity per zone
zone_quantity = joined.groupby('zone')['qty'].sum().reset_index()
#finding the zone with the most items
most_items = zone_quantity.loc[zone_quantity['qty'].idxmax()]
#finding total to account for unmatched vendor
total_zone_qty = zone_quantity['qty'].sum()
missing_qty = joined[joined['zone'].isna()]['qty'].sum()
# printing the information
print("Zone Totals:")
print(zone_quantity)
print("Most items sold:")
print(most_items)

print(f"\nTotal quantity assigned to zones: {total_zone_qty}")
print(f"Difference (unmatched vendor with no zone): {missing_qty} units")

Zone Totals:
  zone  qty
0    A    4
Most items sold:
zone    A
qty     4
Name: 0, dtype: object

Total quantity assigned to zones: 4
Difference (unmatched vendor with no zone): 3 units


### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [6]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                       # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]
# 1. Ensure every order dictionary contains a 'lines' key
cleaned_ragged = []
for order in ragged_json:
    order_copy = order.copy()
    if 'lines' not in order_copy or not order_copy['lines']:
        # Assign a default record with missing item/qty so the order itself isn't lost
        order_copy['lines'] = [{'item': None, 'qty': None}]
    cleaned_ragged.append(order_copy)

# 2. Flatten the payload while preserving order metadata
ragged_df = pd.json_normalize(cleaned_ragged, record_path='lines', meta=['order','vendor_id'])

# Reorder columns for clarity
ragged_df = ragged_df[['order', 'vendor_id', 'item', 'qty']]

print("Ragged DataFrame:")
print(ragged_df)

Ragged DataFrame:
  order vendor_id   item  qty
0     4      V-01   Soda  2.0
1     5      V-10   None  NaN
2     6      V-01  Fries  NaN


### Q6 — Validate

In [7]:
assert len(lines) == 4
assert lines['qty'].sum() == 7
assert len(joined) == len(lines), 'the vendor join changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

One row per line item allows us to get a more coherent sense for the total quantity assigned to a vendor within a specific zone. This makes the question of calculating total quantity per vendor much easier than if it were one row per order.

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

The practical difference is that a missing quantity is data that is unrecorded, unassigned, or just missing, while a zero quantity is an actual quantity. Where conflation can be an issue is if you convert missing values to zero, which falsely records them. This can be done through a function like .fillna(0), which is why we were told not to use that in Q5.

_your answer here_